In [1]:
import os, time, json
import numpy as np
import pandas as pd
import torch
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

DATA_DIR = "/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset"
df = pd.read_csv(f"{DATA_DIR}/train.csv")[["file_name", "label"]]
FAKE, REAL = 1, 0                                   # in this dataset: 1 = AI, 0 = real

# Exactly the same pair split as v2 (same code, same seed), so the comparison is fair
df["pair_id"] = np.arange(len(df)) // 2
rng = np.random.default_rng(42)
pairs = rng.permutation(df.pair_id.unique())
n_train, n_val = int(0.8 * len(pairs)), int(0.1 * len(pairs))
split_of = {}
for p in pairs[:n_train]:                 split_of[p] = "train"
for p in pairs[n_train:n_train + n_val]:  split_of[p] = "val"
for p in pairs[n_train + n_val:]:         split_of[p] = "test"
df["split"] = df.pair_id.map(split_of)

train_df = df[df.split == "train"].reset_index(drop=True)
val_df   = df[df.split == "val"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)
print(f"Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")
assert len(set(train_df.pair_id) & set(test_df.pair_id)) == 0
print("First test files (should match v2's split_v2.csv):", test_df.file_name.head(3).tolist())

Device: cuda Tesla T4
Train 63960 | Val 7994 | Test 7996
First test files (should match v2's split_v2.csv): ['train_data/42b279d1413246e3afeaa551b1502bb8.jpg', 'train_data/886e1c69d7074598bc752912a7d73ff3.jpg', 'train_data/abc69a9e3a834c82a777a2207f1d8bc8.jpg']


In [2]:
from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor
from torch.utils.data import Dataset, DataLoader

MODEL_ID = "openai/clip-vit-large-patch14"
processor = CLIPImageProcessor.from_pretrained(MODEL_ID)      # CLIP's own image recipe
clip = CLIPVisionModelWithProjection.from_pretrained(MODEL_ID).to(device).eval()
for p in clip.parameters():
    p.requires_grad = False                                   # frozen: CLIP never changes

print(f"CLIP image encoder weights: {sum(p.numel() for p in clip.parameters()):,}")
print("Description card length:", clip.config.projection_dim)

class CsvClipDataset(Dataset):
    def __init__(self, frame):
        self.paths = frame.file_name.tolist()
        self.labels = frame.label.tolist()
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(f"{DATA_DIR}/{self.paths[i]}").convert("RGB")
        pixels = processor(images=img, return_tensors="pt")["pixel_values"][0]
        return pixels, self.labels[i]

x, y = CsvClipDataset(train_df.head(2))[0]
print("One prepared image:", x.shape)                         # expect [3, 224, 224]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{

CLIP image encoder weights: 303,966,208
Description card length: 768
One prepared image: torch.Size([3, 224, 224])


In [3]:
def make_cards(frame, name):
    loader = DataLoader(CsvClipDataset(frame), batch_size=128, shuffle=False,
                        num_workers=4, pin_memory=True)
    cards, t0 = [], time.time()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i, (x, _) in enumerate(loader):
            e = clip(pixel_values=x.to(device, non_blocking=True)).image_embeds.float()
            e = e / e.norm(dim=-1, keepdim=True)              # same scale for every card
            cards.append(e.cpu())
            if (i + 1) % 50 == 0:
                print(f"  {name}: {(i + 1) * 128}/{len(frame)} images")
    X = torch.cat(cards).numpy()
    print(f"{name}: {X.shape} in {(time.time() - t0) / 60:.1f} min")
    return X, frame.label.values

X_train, y_train = make_cards(train_df, "train")
X_val,   y_val   = make_cards(val_df,   "val")
X_test,  y_test  = make_cards(test_df,  "test")

  train: 6400/63960 images
  train: 12800/63960 images
  train: 19200/63960 images
  train: 25600/63960 images
  train: 32000/63960 images
  train: 38400/63960 images
  train: 44800/63960 images
  train: 51200/63960 images
  train: 57600/63960 images
  train: 64000/63960 images
train: (63960, 768) in 12.5 min
  val: 6400/7994 images
val: (7994, 768) in 1.6 min
  test: 6400/7996 images
test: (7996, 768) in 1.6 min


In [4]:
from sklearn.linear_model import LogisticRegression

# Try 3 strictness settings, choose using VALIDATION (never test)
sweep, best = [], None
for C in [0.1, 1.0, 10.0]:
    t0 = time.time()
    clf = LogisticRegression(C=C, max_iter=3000)
    clf.fit(X_train, y_train)
    val_acc = (clf.predict(X_val) == y_val).mean()
    sweep.append({"C": C, "val_acc": float(val_acc)})
    print(f"C={C}: val acc {val_acc:.4f} ({time.time() - t0:.0f}s)")
    if best is None or val_acc > best[1]:
        best = (clf, val_acc, C)
clf, best_val_acc, best_C = best
assert clf.classes_.tolist() == [0, 1]                     # so the weights point toward AI

# Final exam, once
p_fake = clf.predict_proba(X_test)[:, 1]                   # column 1 = AI
preds = (p_fake >= 0.5).astype(int)
fake_caught  = int(((preds == FAKE) & (y_test == FAKE)).sum())
fake_missed  = int(((preds == REAL) & (y_test == FAKE)).sum())
real_correct = int(((preds == REAL) & (y_test == REAL)).sum())
real_flagged = int(((preds == FAKE) & (y_test == REAL)).sum())
test_acc = (preds == y_test).mean()
fpr  = real_flagged / (real_flagged + real_correct)
miss = fake_missed / (fake_missed + fake_caught)
print(f"\nBest C={best_C} | TEST ACCURACY {test_acc:.4f} | FPR {fpr:.4f} | miss rate {miss:.4f}")

# The decider: just 768 weights + 1 bias, saved as plain numbers
head = {
    "model_id": MODEL_ID,
    "model_version": "clip-vitl14-linear-v3",
    "weights": clf.coef_[0].tolist(),
    "bias": float(clf.intercept_[0]),
    "normalize": "l2",
    "class_to_idx": {"FAKE": FAKE, "REAL": REAL},
}
with open("/kaggle/working/v3_head.json", "w") as f:
    json.dump(head, f)

results = {
    "approach": "frozen CLIP ViT-L/14 image embeddings + logistic regression",
    "dataset": "alessandrasala79/ai-vs-human-generated-dataset",
    "split": "by pair, 80/10/10, seed 42 (identical to v2)",
    "split_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "C_sweep": sweep, "best_C": best_C, "best_val_acc": float(best_val_acc),
    "test_accuracy": float(test_acc), "false_positive_rate": fpr, "miss_rate": miss,
}
with open("/kaggle/working/results_v3.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved v3_head.json and results_v3.json")

C=0.1: val acc 0.9492 (1s)
C=1.0: val acc 0.9682 (2s)
C=10.0: val acc 0.9714 (2s)

Best C=10.0 | TEST ACCURACY 0.9726 | FPR 0.0278 | miss rate 0.0270
Saved v3_head.json and results_v3.json
